# 第34课 · Nyquist 定理与混叠（aliasing） — 6kHz 正弦波被 8kHz 采样后会变成什么

**今日目标**：理解为什么采样率（sample rate，sr）必须 > 2×最高频率。亲眼看到一个高频正弦被欠采样后“伪装”成低频。

**为什么对Aurora重要**：Aurora 的 16 kHz 音频管道只处理 0–8 kHz 的频率分量；录音前的抗混叠滤波器（anti-aliasing filter）把 8 kHz 以上截断，正是为了防止高频折叠成虚假的低频特征进入 mel 频谱或 MFCC。

← **上一课**　[L33 · 正弦波生成](L33_sine_wave.ipynb)

> 上节课学习了 **正弦波生成**：x[n]=A·sin(2πfn/sr)，亲手实现并对齐 aurora.audio.sine。  
> 本课将探讨 **Nyquist 定理与混叠**。

## 本课剧情：CD 音频为什么选 44100 Hz？

人耳能听到的最高频率约 20 kHz。CD 的采样率是 44100 Hz——刚好是 20000 的两倍多一点。

这不是巧合，而是**奈奎斯特定理（Nyquist theorem）**：要不失真地采样频率为 `f` 的信号，采样率必须满足 `sr > 2f`。否则，高频会"折叠"成一个假的低频——这就是**混叠（aliasing）**。

**直觉**：用 8 kHz 采 6 kHz 正弦波，会发生什么？

采样点之间间隔 `1/8000` 秒。6 kHz 波在每个间隔内转了 `6000/8000 = 0.75` 圈——比 0.5 圈多，但数字化系统"看不出"0.75 圈和 -0.25 圈的区别（都产生同样的采样点序列）。因此 6 kHz 被误认为 `|6000 - 8000·round(6000/8000)| = 2000 Hz`。

折叠公式：

$$\text{alias}(f, f_s) = \left| f - f_s \cdot \text{round}\!\left(\frac{f}{f_s}\right) \right|$$

本节任务：实现 `predict_alias_freq(freq, sample_rate)` 并用 8 kHz/6 kHz 案例验证。

## 1. 概念

采样率 `sr` 能如实表示的最高频率是 `sr/2`（**Nyquist 频率**）。超过它，高频会折叠成一个假的低频——这就是**混叠**。

**为什么会折叠**：采样只记录离散时刻的振幅，丢失了两次采样之间的连续细节。超过 Nyquist 的频率 `f` 与其镜像频率在每个采样点上产生完全相同的数值，因此两者在离散域无法区分。`round(f/sr)` 找到最近的整数倍采样率（即最近的折叠中心 `k·sr`），`abs(f - k·sr)` 计算真实频率到这个折叠中心的距离，得到视在频率。例如：`sr=8000, f=6000` → 折叠中心 `1×8000=8000`，距离 `|6000-8000|=2000 Hz`，6 kHz 听起来像 2 kHz。

## 实验入口：把声音拆成可观察的数组

用 `sr=8000` 和极短时长（0.01 s）生成采样点，让数组足够小，可以直接打印。后续代码会对比 `true_freq=6000` 的高频波和折叠后的低频波，验证两者在离散采样点上完全重合。

In [ ]:
import numpy as np
from aurora.audio import sine
print('就绪')

## 动手观察：序列怎样一步步变成信号

修改 `sample_rate`、`duration`、`freq`，观察打印出的时间轴长度（`N`）和采样点间距（`1/sr`）如何随之变化。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：把所有频率都折回 Nyquist 范围

这个实验遍历 0–20 Hz 的频率，用 `|f - sr·round(f/sr)|` 计算每个频率在 `sr=10` 时的视在频率，展示 Nyquist 两侧的频率如何对称折叠。

In [ ]:
import numpy as np

sample_rate = 10
nyquist = sample_rate / 2
freqs = np.arange(0, 21)

for f in freqs:
    alias = abs(((f + nyquist) % sample_rate) - nyquist)
    print(f'true={f:2d}Hz -> alias={alias:4.1f}Hz')


## 2. ✏️ 实现混叠频率预测 `predict_alias_freq`

**手算验证（sr=8000）**：

| 真实频率 f | round(f/sr) | f - k·sr | alias |
|---|---|---|---|
| 0 | 0 | 0 | **0 Hz** |
| 2000 | 0 | 2000 | **2000 Hz**（低于 Nyquist，不折叠） |
| 4000 | round(0.5)=0 | 4000 | **4000 Hz**（恰好 Nyquist） |
| 6000 | 1 | -2000 | **2000 Hz**（折叠！） |
| 8000 | 1 | 0 | **0 Hz**（正好一圈，混叠为 DC） |
| 10000 | 1 | 2000 | **2000 Hz** |

规律：频率关于 Nyquist（4000 Hz）镜像折叠，每隔 `sr` 重复一次。

> **实现提示**：`abs(freq - sample_rate * round(freq / sample_rate))`  
> 注意 Python `round(0.5)=0`（银行家舍入），与 NumPy `np.round(0.5)=0.0` 一致。

### 写代码前，先把变量表补完整

写 `predict_alias_freq` 前明确三件事：
- 输入：`freq`（待测频率，Hz）和 `sample_rate`（采样率，Hz）
- 关键步骤：计算 `k = round(freq / sample_rate)`，再求 `abs(freq - k * sample_rate)`
- 返回：float，即该频率的视在频率（范围 0 到 `sample_rate / 2`）

In [ ]:
def predict_alias_freq(freq, sample_rate):
    """返回 freq 被 sample_rate 采样后的视在频率（0 到 Nyquist 范围）。
    折叠公式：alias = abs(freq - sample_rate * round(freq / sample_rate))
    """
    # ✏️ TODO: 实现折叠公式
    raise NotImplementedError(
        "predict_alias_freq: 请实现折叠公式 abs(freq - sr * round(freq/sr))"
    )


In [ ]:
# 边界条件验证 — 实现 predict_alias_freq 后运行此格
# 这三个边界恰好是最容易弄错符号或取整方向的地方
try:
    assert predict_alias_freq(0, 8000) == 0.0,    "freq=0 应返回 0.0"
    assert predict_alias_freq(4000, 8000) == 4000.0, "freq=sr/2 应返回 Nyquist 4000.0"
    assert predict_alias_freq(8000, 8000) == 0.0,  "freq=sr 应折回 0.0"
    print('✅ 边界条件全部通过')
except NotImplementedError:
    print('⚠️  请先实现 predict_alias_freq，再运行此格')


sr = 8000          # Nyquist = 4000 Hz
true_freq = 6000   # 高于 Nyquist

try:
    alias = predict_alias_freq(true_freq, sr)
except NotImplementedError as _e:
    print(f'⚠️  尚未实现 predict_alias_freq：{_e}')
    print('  → 请完成上方 TODO 后重新运行此格。')
    raise SystemExit(0)  # 友好停止，避免后续 NameError 级联崩溃

print('预测视在频率:', alias, 'Hz')
assert 0 <= alias <= sr/2, '视在频率应落在 0..Nyquist'

x_aliased = sine(true_freq, duration=0.01, sample_rate=sr)
x_lowfreq = sine(alias,     duration=0.01, sample_rate=sr)
diff = float(np.max(np.abs(x_aliased - x_lowfreq)))
print('采样点与低频版差异:', f'{diff:.2e}', '(越接近0越说明混叠成立)')
assert diff < 1e-10, f'混叠验证失败：差值 {diff:.2e} 应 < 1e-10'
print('✅ 混叠验证通过：6 kHz 的采样点与 2 kHz 纯音完全重合')


In [ ]:
sr = 8000          # Nyquist = 4000 Hz
true_freq = 6000   # 高于 Nyquist
alias = predict_alias_freq(true_freq, sr)
print('预测视在频率:', alias, 'Hz')
assert 0 <= alias <= sr/2, '视在频率应落在 0..Nyquist'

x_aliased = sine(true_freq, duration=0.01, sample_rate=sr)
x_lowfreq = sine(alias,     duration=0.01, sample_rate=sr)
diff = float(np.max(np.abs(x_aliased - x_lowfreq)))
print('采样点与低频版差异:', f'{diff:.2e}', '(越接近0越说明混叠成立)')

## 4. 画图：高频采样点 vs 那个假低频，几乎重合

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(x_aliased, 'o-', ms=4, label=f'sampled {true_freq} Hz @ {sr} Hz')
plt.plot(x_lowfreq, 'x--', ms=5, label=f'pure {alias:.0f} Hz')
plt.title('Aliasing: a 6 kHz tone masquerades as a low tone')
plt.xlabel('sample n'); plt.legend(); plt.tight_layout(); plt.show()

**结论**：欠采样让高频伪装成低频。这就是录音前要"抗混叠滤波"的原因。

**🎉 完成后**：`git commit -m 'learn: L34 aliasing'`

## 🎨 图示：混叠——6kHz 被 8kHz 采样后伪装成低频

In [ ]:
import aurora.aviz as aviz; aviz.style()
aviz.aliasing(6000, 8000);

In [ ]:
sr = 12
nyq = sr / 2
for f in [1, 5, 6, 7, 11, 13, 17]:
    alias = abs(((f + nyq) % sr) - nyq)
    status = '安全' if f <= nyq else '伪装'
    print(f'{f:>2}Hz -> {alias:>4.1f}Hz  {status}')


## 参数实验：扫描 0–32000 Hz，观察折叠规律

固定 `sr=16000`，把 `freq` 从 0 扫到 32000，每 1000 Hz 打印 `predict_alias_freq`，观察结果在 0–8000 Hz 范围内折叠：16000 Hz 折回 0 Hz，24000 Hz 折回 8000 Hz，32000 Hz 再次折回 0 Hz。把下面代码粘贴到空白单元格运行：

```python
sr = 16000
for f in range(0, 33000, 1000):
    alias = predict_alias_freq(f, sr)
    print(f'{f:>6} Hz  ->  {alias:>6.0f} Hz')
```

规律：折叠图案每隔 `sr` Hz 重复一次；Nyquist（8000 Hz）是对称轴，9000 Hz 和 7000 Hz 产生相同的视在频率（都是 1000 Hz）。

## 本课收束

现在可以用 `predict_alias_freq(freq, sr)` 把任意频率映射到 0–Nyquist 范围内的视在频率。画图验证会显示：6 kHz 高频的采样点与 2 kHz 低频完全重合，在离散域无法区分。Aurora 音频管道在 16 kHz 采样率下只提取 0–8 kHz 的 mel 特征；8 kHz 以上的能量若未被抗混叠滤波器截断，会以假低频的形式污染模型输入。下一节（L35，欧拉公式与旋转因子）引入复数表示，为 DFT 推导奠基；L36 再引入汉宁窗压制频谱泄漏。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))


---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：混叠手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：sr=8000 Hz，f=6000 Hz
- alias = |f - sr·round(f/sr)| = ?（步骤：round(6000/8000)=?，然后算）

**问 2**：sr=8000 Hz，f=3000 Hz（低于 Nyquist=4000）
- alias = ?（应该不折叠）

**问 3**：sr=8000 Hz，f=9000 Hz
- alias = ?（比 sr 还高，折叠两次？）

**问 4**：为什么 CD（44100 Hz）能录 20 kHz 人声上限？
- Nyquist 要求 sr > 2·f_max，即 sr > ? Hz
- 44100/2 = ? Hz（是否超过 20000？）

**问 5**：Aurora 音频管道：如果模型输入信号上限 8 kHz，采样率最低需要多少？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：sr=8000, f=6000 → alias=2000
sr = 8000
assert np.isclose(abs(6000 - sr * round(6000/sr)), 2000.0)
try:
    a1 = predict_alias_freq(6000, sr)
    assert np.isclose(a1, 2000.0), f"alias(6000,8000) 应=2000，得到 {a1}"
    print(f"Q1 ✅  alias(6000, 8000) = {a1} Hz（6kHz折叠为2kHz）")
except NotImplementedError:
    print("⬜ Q1：请先实现 predict_alias_freq()，再运行对答案格")

# 问2：sr=8000, f=3000 → alias=3000（不折叠）
assert np.isclose(abs(3000 - sr * round(3000/sr)), 3000.0)
try:
    a2 = predict_alias_freq(3000, sr)
    assert np.isclose(a2, 3000.0)
    print(f"Q2 ✅  alias(3000, 8000) = {a2} Hz（低于Nyquist=4000，不折叠）")
except NotImplementedError:
    print("⬜ Q2：请先实现 predict_alias_freq()，再运行对答案格")

# 问3：sr=8000, f=9000 → alias=1000
assert np.isclose(abs(9000 - sr * round(9000/sr)), 1000.0)
try:
    a3 = predict_alias_freq(9000, sr)
    assert np.isclose(a3, 1000.0)
    print(f"Q3 ✅  alias(9000, 8000) = {a3} Hz（round(9000/8000)=1，|9000-8000|=1000）")
except NotImplementedError:
    print("⬜ Q3：请先实现 predict_alias_freq()，再运行对答案格")

# 问4：CD 44100 Hz
nyquist_cd = 44100 / 2
assert nyquist_cd == 22050.0 and nyquist_cd > 20000
print(f"Q4 ✅  44100/2 = {nyquist_cd} Hz > 20000 Hz（人耳上限），Nyquist 满足")

# 问5：Aurora 8kHz 上限 → sr > 16000
min_sr = 2 * 8000
print(f"Q5 ✅  最低采样率 = 2×8000 = {min_sr} Hz，Aurora 常用 16000 Hz 正好满足")
print("\n🎉 混叠白板挑战通过！折叠公式 |f - sr·round(f/sr)| 已内化。")

In [ ]:
# ✏️ 本课自评
l34_review = {
    "nyquist_theorem":           None,  # 记住 sr > 2f_max 的奈奎斯特条件？True/False
    "alias_formula":             None,  # 记住 alias=|f - sr·round(f/sr)|？True/False
    "predict_alias_implemented": None,  # predict_alias_freq 实现并通过断言？True/False
    "cd_sampling_rate":          None,  # 理解44100Hz为何能覆盖20kHz人耳上限？True/False
    "whiteboard_passed":         None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l34_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l34_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L34 全部通关！进入 L35：欧拉公式遇见 FFT')

---

→ **下一课**　[L35 · 欧拉公式遇见 FFT](L35_euler_fft.ipynb)

> 下节课将学习 **欧拉公式遇见 FFT**：e^{-2πikn/N} 是什么，旋转因子可视化。